# Comparación de modelos para clasificación de mora

Se comparan regresión logística, árbol de decisión, Random Forest, Extra Trees, HistGradientBoosting y GradientBoosting. El criterio de selección es F1 macro en validation; el conjunto test se utiliza una sola vez para reportar el resultado final del ganador. Se excluyen `ID`, `fecha_reporte` y `atraso` para reducir fuga directa de información. Cada línea ejecutable incluye un comentario en español.

## 1. Cargar los CSV limpios en Colab

In [ ]:
from google.colab import files  # Importa el selector de archivos de Google Colab.
archivos_subidos = files.upload()  # Permite cargar train, validation y test limpios.

## 2. Importar librerías

In [ ]:
import pandas as pd  # Importa pandas para gestionar los datos tabulares.
import numpy as np  # Importa NumPy para cálculos numéricos.
import matplotlib.pyplot as plt  # Importa Matplotlib para las gráficas.
import seaborn as sns  # Importa Seaborn para las matrices de confusión.
import time  # Importa time para registrar duración de ajuste.
from pathlib import Path  # Importa Path para crear carpetas y rutas.
import joblib  # Importa joblib para guardar el modelo ganador.
from sklearn.compose import ColumnTransformer  # Importa el transformador por tipo de variable.
from sklearn.preprocessing import StandardScaler, OneHotEncoder, label_binarize  # Importa escalado, codificación y binarización.
from sklearn.linear_model import LogisticRegression  # Importa la regresión logística.
from sklearn.tree import DecisionTreeClassifier  # Importa el árbol de decisión.
from sklearn.ensemble import RandomForestClassifier, ExtraTreesClassifier, HistGradientBoostingClassifier, GradientBoostingClassifier  # Importa modelos de ensamble y boosting.
from sklearn.metrics import accuracy_score, balanced_accuracy_score, precision_score, recall_score, f1_score, classification_report, confusion_matrix, roc_curve, auc, precision_recall_curve, average_precision_score  # Importa métricas y herramientas para las curvas.
from sklearn.utils.class_weight import compute_sample_weight  # Importa el cálculo de pesos para clases desbalanceadas.

## 3. Leer y verificar las particiones

In [ ]:
rutas = {'train': 'data_train-mora_limpio.csv', 'validation': 'data_validation-mora_limpio.csv', 'test': 'data_test-mora_limpio.csv'}  # Asocia nombres de conjuntos y archivos.
datos = {nombre: pd.read_csv(ruta, dtype={'ID': 'string'}) for nombre, ruta in rutas.items()}  # Lee cada CSV y conserva el ID como texto.
for nombre, tabla in datos.items():  # Revisa todas las particiones cargadas.
    print(nombre, tabla.shape, 'faltantes=', int(tabla.isna().sum().sum()), 'duplicados=', int(tabla.duplicated().sum()))  # Informa tamaño y calidad básica.

## 4. Definir variables y prevenir fuga directa

In [ ]:
objetivo = 'mora'  # Define la etiqueta que se desea clasificar.
excluidas = ['ID', 'fecha_reporte', 'atraso', objetivo]  # Excluye identificador, fecha y días de atraso observados.
variables = [c for c in datos['train'].columns if c not in excluidas]  # Forma la lista de predictores.
X = {s: d[variables].copy() for s, d in datos.items()}  # Separa los predictores para cada conjunto.
y = {s: d[objetivo].astype(str) for s, d in datos.items()}  # Separa las etiquetas de cada conjunto.
clases = sorted(y['train'].unique())  # Fija el orden de categorías usando train.
numericas = X['train'].select_dtypes(include='number').columns.tolist()  # Identifica predictores numéricos.
categoricas = [c for c in variables if c not in numericas]  # Identifica predictores categóricos.
print('Clases:', clases)  # Muestra las categorías objetivo.
print('Variables numéricas:', len(numericas), '| Variables categóricas:', len(categoricas))  # Resume el tipo de predictores.

## 5. Preprocesar con train y transformar los otros conjuntos

In [ ]:
preprocesador = ColumnTransformer([('numericas', StandardScaler(), numericas), ('categoricas', OneHotEncoder(handle_unknown='ignore', sparse_output=False), categoricas)], remainder='drop', sparse_threshold=0)  # Escala números y codifica texto con salida densa para todos los modelos.
preprocesador.fit(X['train'])  # Ajusta escalas y categorías únicamente con train.
X_transformado = {s: preprocesador.transform(X[s]) for s in X}  # Aplica sin reajustar a train, validation y test.
pesos_train = compute_sample_weight(class_weight='balanced', y=y['train'])  # Calcula pesos de entrenamiento para compensar desbalance en boosting.

## 6. Definir modelos candidatos
Los modelos usan pesos balanceados cuando su implementación lo permite. En los dos boosting se usan pesos por observación. Los parámetros son deliberadamente moderados para mantener una comparación reproducible.

In [ ]:
candidatos = {  # Crea el diccionario de algoritmos y configuraciones.
    'Regresión logística (baseline)': LogisticRegression(max_iter=2500, class_weight='balanced', solver='lbfgs', C=1.0, random_state=42),  # Define la regresión logística inicial balanceada.
    'Árbol de decisión': DecisionTreeClassifier(class_weight='balanced', min_samples_leaf=8, max_depth=14, random_state=42),  # Define árbol limitado para moderar sobreajuste.
    'Random Forest': RandomForestClassifier(n_estimators=350, class_weight='balanced_subsample', min_samples_leaf=2, max_features='sqrt', n_jobs=-1, random_state=42),  # Define bosque aleatorio balanceado por muestra.
    'Extra Trees': ExtraTreesClassifier(n_estimators=350, class_weight='balanced', min_samples_leaf=2, max_features='sqrt', n_jobs=-1, random_state=42),  # Define árboles extremadamente aleatorizados.
    'HistGradientBoosting': HistGradientBoostingClassifier(max_iter=180, max_leaf_nodes=31, learning_rate=0.08, l2_regularization=1.0, early_stopping=True, random_state=42),  # Define boosting por histogramas con regularización.
    'GradientBoosting': GradientBoostingClassifier(n_estimators=120, learning_rate=0.08, max_depth=3, min_samples_leaf=8, random_state=42)  # Define gradient boosting clásico.
}  # Cierra la lista de candidatos.

## 7. Entrenar en train y comparar en validation
El ranking se calcula exclusivamente en validation. El mejor se elige primero por F1 macro y luego por balanced accuracy.

In [ ]:
modelos_ajustados = {}  # Crea un diccionario para conservar candidatos entrenados.
filas_resultados = []  # Crea una lista para almacenar métricas y tiempos.
for nombre, estimador in candidatos.items():  # Recorre cada algoritmo.
    inicio = time.time()  # Registra el instante inicial del ajuste.
    if nombre in ['HistGradientBoosting', 'GradientBoosting']:  # Comprueba si el estimador acepta pesos por observación.
        estimador.fit(X_transformado['train'], y['train'], sample_weight=pesos_train)  # Ajusta boosting con pesos balanceados.
    else:  # Atiende algoritmos con ponderación interna de clases.
        estimador.fit(X_transformado['train'], y['train'])  # Ajusta el estimador usando únicamente train.
    pred = estimador.predict(X_transformado['validation'])  # Predice las clases de validation.
    modelos_ajustados[nombre] = estimador  # Conserva el candidato entrenado.
    filas_resultados.append({'modelo': nombre, 'accuracy': accuracy_score(y['validation'], pred), 'balanced_accuracy': balanced_accuracy_score(y['validation'], pred), 'precision_macro': precision_score(y['validation'], pred, average='macro', zero_division=0), 'recall_macro': recall_score(y['validation'], pred, average='macro', zero_division=0), 'f1_macro': f1_score(y['validation'], pred, average='macro', zero_division=0), 'f1_weighted': f1_score(y['validation'], pred, average='weighted', zero_division=0), 'segundos_entrenamiento': time.time() - inicio})  # Calcula y guarda medidas comparables.
comparacion = pd.DataFrame(filas_resultados).sort_values(['f1_macro', 'balanced_accuracy'], ascending=False).reset_index(drop=True)  # Ordena por el criterio acordado.
display(comparacion)  # Presenta la tabla de resultados de validation.
mejor_nombre = comparacion.iloc[0]['modelo']  # Selecciona el modelo con mejor F1 macro en validation.
mejor_modelo = modelos_ajustados[mejor_nombre]  # Recupera el estimador seleccionado.
print('Modelo seleccionado:', mejor_nombre)  # Informa el ganador antes de evaluar test.

## 8. Graficar comparación de modelos en validation

In [ ]:
for metrica, titulo, archivo in [('f1_macro', 'F1 macro en validation', 'F1 macro'), ('balanced_accuracy', 'Balanced accuracy en validation', 'Balanced accuracy')]:  # Recorre métricas para comparar los algoritmos.
    ordenado = comparacion.sort_values(metrica)  # Ordena los modelos por la métrica actual.
    colores = ['#e59a35' if n == mejor_nombre else '#5480a8' for n in ordenado['modelo']]  # Resalta el ganador de validation.
    fig, ax = plt.subplots(figsize=(11, 6))  # Crea un lienzo para la comparación.
    ax.barh(ordenado['modelo'], ordenado[metrica], color=colores)  # Dibuja una barra por algoritmo.
    ax.set_xlim(0, 1)  # Fija límites comparables entre métricas.
    ax.set_xlabel(archivo)  # Etiqueta el eje horizontal.
    ax.set_title(titulo)  # Añade el título de la gráfica.
    fig.tight_layout()  # Ajusta los espacios de la figura.
    plt.show()  # Presenta la comparación en Colab.

## 9. Evaluación final del ganador en test
Una vez elegido el modelo con validation, se mide el ganador en test. Evite escoger otro modelo después de observar el resultado de test.

In [ ]:
pred_test = mejor_modelo.predict(X_transformado['test'])  # Predice test una vez seleccionado el candidato.
prob_test = mejor_modelo.predict_proba(X_transformado['test'])  # Obtiene probabilidades para las gráficas ROC y PR.
metricas_test = {'accuracy': accuracy_score(y['test'], pred_test), 'balanced_accuracy': balanced_accuracy_score(y['test'], pred_test), 'precision_macro': precision_score(y['test'], pred_test, average='macro', zero_division=0), 'recall_macro': recall_score(y['test'], pred_test, average='macro', zero_division=0), 'f1_macro': f1_score(y['test'], pred_test, average='macro', zero_division=0), 'f1_weighted': f1_score(y['test'], pred_test, average='weighted', zero_division=0)}  # Calcula medidas finales en test.
print('Resultado final en test:', metricas_test)  # Informa métricas del ganador.
print(classification_report(y['test'], pred_test, labels=clases, zero_division=0))  # Detalla precision, recall y F1 para cada categoría.

## 10. Graficar F1 por clase y matrices de confusión

In [ ]:
pred_val = mejor_modelo.predict(X_transformado['validation'])  # Predice validation para comparar F1 por categoría.
reporte_val = classification_report(y['validation'], pred_val, labels=clases, output_dict=True, zero_division=0)  # Calcula métricas por clase en validation.
reporte_test = classification_report(y['test'], pred_test, labels=clases, output_dict=True, zero_division=0)  # Calcula métricas por clase en test.
pos = np.arange(len(clases))  # Crea posiciones para las barras agrupadas.
fig, ax = plt.subplots(figsize=(12, 6))  # Crea el lienzo de F1 por clase.
ax.bar(pos - 0.19, [reporte_val[c]['f1-score'] for c in clases], 0.38, label='Validation')  # Grafica F1 por clase en validation.
ax.bar(pos + 0.19, [reporte_test[c]['f1-score'] for c in clases], 0.38, label='Test')  # Grafica F1 por clase en test.
ax.set_xticks(pos, clases, rotation=25, ha='right')  # Añade etiquetas legibles de categorías.
ax.set_ylim(0, 1.05)  # Ajusta el eje vertical al rango de puntuaciones.
ax.set_ylabel('F1 por clase')  # Etiqueta el eje vertical.
ax.set_title(f'F1 por categoría — {mejor_nombre}')  # Titula la comparación por categoría.
ax.legend()  # Muestra leyenda de particiones.
fig.tight_layout()  # Ajusta márgenes.
plt.show()  # Muestra el gráfico de F1.
cm = confusion_matrix(y['test'], pred_test, labels=clases)  # Calcula conteos de test.
cm_norm = confusion_matrix(y['test'], pred_test, labels=clases, normalize='true')  # Calcula proporciones por categoría real.
fig, axes = plt.subplots(1, 2, figsize=(18, 7))  # Crea dos paneles de matrices.
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues', xticklabels=clases, yticklabels=clases, ax=axes[0])  # Visualiza conteos.
sns.heatmap(cm_norm, annot=True, fmt='.2f', cmap='Greens', xticklabels=clases, yticklabels=clases, ax=axes[1])  # Visualiza tasas por clase real.
axes[0].set_title('Conteos de test')  # Titula primer panel.
axes[1].set_title('Proporciones por clase real')  # Titula segundo panel.
for eje in axes:  # Recorre ambos ejes.
    eje.set_xlabel('Predicción')  # Etiqueta el eje horizontal.
    eje.set_ylabel('Clase real')  # Etiqueta el eje vertical.
    eje.tick_params(axis='x', rotation=35)  # Gira categorías en eje X.
fig.tight_layout()  # Ajusta el espacio de los paneles.
plt.show()  # Muestra matrices de confusión.

## 11. Curvas ROC y precisión-recobrado del ganador en test

In [ ]:
y_bin = label_binarize(y['test'], classes=clases)  # Convierte las clases de test a formato uno contra resto.
fig, ax = plt.subplots(figsize=(9, 7))  # Crea la figura ROC.
for i, clase in enumerate(clases):  # Recorre categorías.
    fpr, tpr, _ = roc_curve(y_bin[:, i], prob_test[:, i])  # Calcula ROC para una categoría.
    ax.plot(fpr, tpr, label=f'{clase} AUC={auc(fpr, tpr):.3f}')  # Grafica curva ROC y AUC.
ax.plot([0, 1], [0, 1], 'k--', label='Azar')  # Añade la referencia aleatoria.
ax.set_title('Curvas ROC uno contra resto — test')  # Titula la gráfica ROC.
ax.set_xlabel('Tasa de falsos positivos')  # Etiqueta el eje X.
ax.set_ylabel('Tasa de verdaderos positivos')  # Etiqueta el eje Y.
ax.legend(fontsize=8)  # Muestra AUC por clase.
fig.tight_layout()  # Ajusta la figura.
plt.show()  # Presenta ROC.
fig, ax = plt.subplots(figsize=(9, 7))  # Crea la figura precisión-recobrado.
for i, clase in enumerate(clases):  # Recorre cada categoría.
    precision, recobrado, _ = precision_recall_curve(y_bin[:, i], prob_test[:, i])  # Calcula precisión y recobrado.
    ap = average_precision_score(y_bin[:, i], prob_test[:, i])  # Calcula promedio de precisión.
    ax.plot(recobrado, precision, label=f'{clase} AP={ap:.3f}')  # Dibuja la curva y su AP.
ax.set_title('Precisión-recobrado uno contra resto — test')  # Titula la figura.
ax.set_xlabel('Recobrado')  # Etiqueta el eje horizontal.
ax.set_ylabel('Precisión')  # Etiqueta el eje vertical.
ax.legend(fontsize=8)  # Muestra leyenda.
fig.tight_layout()  # Ajusta márgenes.
plt.show()  # Presenta la curva PR.

## 12. Guardar tabla, métricas y modelo ganador

In [ ]:
carpeta = Path('resultados_comparacion_mora')  # Define la carpeta de salida.
carpeta.mkdir(exist_ok=True)  # Crea la carpeta si no existe.
comparacion.to_csv(carpeta / 'comparacion_validation.csv', index=False, encoding='utf-8-sig')  # Guarda el ranking de validation.
pd.DataFrame([{'modelo': mejor_nombre, **metricas_test}]).to_csv(carpeta / 'resultado_final_test.csv', index=False, encoding='utf-8-sig')  # Guarda las métricas finales del ganador.
pd.DataFrame(classification_report(y['test'], pred_test, labels=clases, output_dict=True, zero_division=0)).T.to_csv(carpeta / 'reporte_por_clase_test.csv', encoding='utf-8-sig')  # Guarda el reporte por categoría.
pd.DataFrame({'real': y['test'], 'prediccion': pred_test, 'probabilidad_maxima': prob_test.max(axis=1)}).to_csv(carpeta / 'predicciones_test.csv', index=False, encoding='utf-8-sig')  # Guarda predicciones y confianza máxima.
joblib.dump({'preprocesador': preprocesador, 'modelo': mejor_modelo, 'variables': variables, 'clases': clases, 'nombre_modelo': mejor_nombre}, carpeta / 'modelo_ganador_mora.joblib')  # Guarda modelo junto al transformador y esquema de variables.
print('Archivos guardados en:', carpeta.resolve())  # Muestra la ruta de resultados.

## Observaciones metodológicas
El ganador se selecciona por F1 macro en validation, que da el mismo peso a cada clase. Confirme que variables como `mora_max_*` estén disponibles antes del instante real de predicción; si contienen información posterior, deben excluirse. Las bases parecen sintéticas, por lo que estas métricas no demuestran desempeño en una cartera bancaria real.